# Homework 01 — Distributions

**Lesson:** `lessons/stats/01_distributions.ipynb`  
**Solutions:** `homework/solutions/01_distributions_hw_solutions.ipynb`

Work through each exercise in order. Some are computation, some are interpretation. Write your answers to written questions in a markdown cell below the exercise.

---

In [1]:
# Imports — don't modify this cell
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

---

## Exercise 1 — Log vs. Simple Returns (Computation)

Download **QQQ** (Nasdaq-100 ETF) daily data from **2018-01-01 to 2024-01-01**.

Compute both simple and log returns. Then answer:

a) On which single day was the **largest discrepancy** between simple and log returns? Print the date, both return values, and the difference.

b) What kind of market event do you think caused that day? (Check the date — you probably know what it was.)

**Hint:** The discrepancy is largest on the most extreme return days.

In [11]:
# Your code here
qqq = yf.download('QQQ', start='2018-01-01', end='2024-01-01', auto_adjust=True)

prices = qqq['Close'].squeeze().dropna()

simple_returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

# print(simple_returns.head())

# they are scalars (arrays), in numpy we can easily do this operation on them
difference = simple_returns - log_returns

# print(difference.head())

# Hint 2: To find the index (date) of the largest absolute value in a Series, look up Series.abs() and Series.idxmax() — chain them together.

idx = difference.abs().idxmax()

print(idx)
# 2020-03-16

[*********************100%***********************]  1 of 1 completed

2020-03-16 00:00:00


*Your written answer for (b):* 2020-03-16 - Let's google this. Okay! It was due to COVID-19.

---

## Exercise 2 — Cross-Asset Fat Tail Comparison (Computation + Interpretation)

Download daily data from **2015-01-01 to 2024-01-01** for these four tickers:

| Ticker | Asset |
|---|---|
| `SPY` | S&P 500 ETF |
| `GLD` | Gold ETF |
| `TLT` | 20+ Year Treasury Bond ETF |
| `BTC-USD` | Bitcoin |

For each asset, compute daily log returns and calculate:
- Annualized mean return
- Annualized volatility
- Skewness
- Excess kurtosis

Print a summary table. Then answer:

a) Which asset has the fattest tails? Does that surprise you?

b) Which asset has the most negative skew? What does that mean for someone holding that asset?

c) Bitcoin has very high volatility. Does it also have the highest excess kurtosis? Why might that be the case or not?

**Hint:** Use `pd.DataFrame` to build the summary table with each ticker as a row.

In [20]:
# exercise 2: Your code here
dates = ['2015-01-01', '2024-01-01']
TRADING_DAYS_IN_YEAR=252

spy = yf.download('SPY', start=dates[0], end=dates[1], auto_adjust=True)
gld = yf.download('GLD', start=dates[0], end=dates[1], auto_adjust=True)
tlt = yf.download('TLT', start=dates[0], end=dates[1], auto_adjust=True)
btcusd = yf.download('BTC-USD', start=dates[0], end=dates[1], auto_adjust=True)

spy_prices = spy['Close'].squeeze().dropna()
gld_prices = gld['Close'].squeeze().dropna()
tlt_prices = tlt['Close'].squeeze().dropna()
btcusd_prices = btcusd['Close'].squeeze().dropna()

def log_returns(prices):
    return np.log(prices / prices.shift(1)).dropna()

spy_log_returns = log_returns(spy_prices)
gld_log_returns = log_returns(gld_prices)
tlt_log_returns = log_returns(tlt_prices)
btcusd_log_returns = log_returns(btcusd_prices)


'''
calculate:

Annualized mean return
Annualized volatility
Skewness
Excess kurtosis
'''

mapped_tickers_returns = [
    ('SPY', spy_log_returns), 
    ('GLD', gld_log_returns), 
    ('TLT', tlt_log_returns),
    ('BTC-USD', btcusd_log_returns),
]

results = dict()

for name, returns in mapped_tickers_returns:
    results[name] = {
        'Annualized mean return': returns.mean() * TRADING_DAYS_IN_YEAR,
        'Annualized volatility': returns.std() * np.sqrt(TRADING_DAYS_IN_YEAR),
        'Skewness': stats.skew(returns),
        'Excess kurtosis': stats.kurtosis(returns),
        
    }


pd.DataFrame(results).T

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,Annualized mean return,Annualized volatility,Skewness,Excess kurtosis
SPY,0.111388,0.181592,-0.793950,13.477772
GLD,0.057489,0.140188,-0.114740,3.036889
TLT,-0.004593,0.154323,0.033771,4.718004
BTC-USD,0.375894,0.593526,-0.793558,11.603576


*Your written answers for (a), (b), (c):*

a) Which asset has the fattest tails? Does that surprise you?

> SPY does. It does surprise me! I'd think that Bitcoin would have the fattest tails. It makes sense that BTC-USD does. But not intuitively SPY. It's probably due to COVID.

b) Which asset has the most negative skew? What does that mean for someone holding that asset?

> SPY does. BTC-USD is very close 2nd. It means they'll see drawdowns that are high, right? Not draw-ups, because it's NEGATIVE skew, not positive skew.

c) Bitcoin has very high volatility. Does it also have the highest excess kurtosis? Why might that be the case or not?

> BTC-USD does not have the highest excess kurtosis. That would be SPY, which is likely due to COVID.

---

## Exercise 3 — The Cost of Assuming Normality (Computation)

Using SPY daily log returns from **2015-01-01 to 2024-01-01**:

a) Fit both a **normal** and a **Student's t** distribution to the returns.

b) Using each model, compute the estimated probability of a **single day loss worse than -5%**.

c) Based on your model predictions, how many such days would you *expect* in a 10-year period (2,520 trading days)? How many actually *occurred* in the data?

d) Which model is closer to reality?

**Hint:** Use `stats.norm.cdf` and `stats.t.cdf` for the probability calculations. Remember log returns are negative for losses.

In [ ]:
# Your code here


*Your written answer for (d):*

---

## Exercise 4 — Volatility Regimes (Computation + Interpretation)

Using SPY daily log returns from **2015-01-01 to 2024-01-01**:

Compute **21-day rolling realized volatility** (annualized). Then:

a) Split the data into two regimes:
   - **Low vol:** days where rolling vol is below its median
   - **High vol:** days where rolling vol is at or above its median

b) For each regime separately, compute skewness and excess kurtosis of the returns.

c) Plot two histograms side by side — one per regime — overlaid with a normal fit.

d) Answer: Are fat tails evenly distributed across regimes, or concentrated in one? What does this imply for position sizing?

In [ ]:
# Your code here


*Your written answer for (d):*

---

## Exercise 5 — Monte Carlo Intuition (Computation + Interpretation)

Simulate **1,000 price paths** for a hypothetical stock over **252 trading days** using these parameters:

- Starting price: $100
- Daily drift (μ): 0.0003
- Daily volatility (σ): 0.012

a) Plot all 1,000 paths on a single chart (use low alpha so it's readable).

b) From the final price distribution, compute:
   - Probability of ending **above $110** (up >10%)
   - Probability of ending **below $90** (down >10%)
   - The 5th percentile outcome (worst 5% of scenarios)

c) Now repeat the simulation but **double the volatility** (σ = 0.024), keeping everything else the same. How do the three probabilities change?

d) Answer: Doubling volatility doesn't change the *expected* final price, but what does it do to the *range* of outcomes? Why does this matter for a trader?

**Hint:** Use `np.random.seed(42)` for reproducibility.

In [ ]:
# Your code here


*Your written answer for (d):*

---

## Bonus — The Turkey Problem

This is a conceptual exercise — no code required.

Nassim Taleb's *Black Swan* describes the "turkey problem": a turkey is fed every day for 1,000 days and concludes, based on all available data, that humans are friendly and food arrives reliably. On day 1,001 — Thanksgiving — it is proven catastrophically wrong.

Answer the following:

a) How does this map to fitting a normal distribution to SPY returns using 2010–2019 data and then using that model in March 2020?

b) A strategy's backtest shows a Sharpe ratio of 2.1 with no losing months over 3 years of data. Should you trust this? What distribution-related concern applies?

c) What is one practical thing a quant can do to partially defend against turkey-problem-style model failure?

*Your written answers:*